# Insurance 04 — Real XLSX round-trip at 10,000 and 100,000 rows

## What this notebook proves
These are genuine Excel workbooks, created with the explicitly approved **openpyxl** fallback. Each workbook contains five readable sheets: Readme, Claims, Policies, DossierLinks and DataDictionary. The inputs are entirely synthetic, not real insurer records.

We test the complete path: **JSONL source → XLSX workbook → streamed worksheet rows → validated SQL tables → exhaustive scoped aggregates**. Every imported row is compared with the source; totals alone could hide offsetting errors.

We do not vectorize accounting cells. RAG is useful for contractual explanations; SQL answers exhaustive analytical questions. This notebook does not claim that the combined workflow is already exposed in the web application.

In [1]:
from pathlib import Path
from contextlib import closing
import sys, os, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/app').is_dir())
if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))
import pandas as pd
from IPython.display import display
from ingestion.insurance_excel import export_excel, benchmark_excel, load_excel, verify_dossier_links
from ingestion.insurance_dossiers import generate_dossiers
BASE = ROOT / 'data/insurance_v2/synthetic'
sizes = [int(x) for x in os.getenv('INSURANCE_EXCEL_ROWS', '10000,100000').split(',')]
print('Actual requested row counts:', sizes)
print('API/model calls: none')

Actual requested row counts: [10000, 100000]
API/model calls: none


## 1. Generate reusable, protected source files
Write-only export limits workbook object memory. Read-only import streams source rows. SQLite is in memory for this experiment; its storage, indexes and sorting still scale with the row count.

Existing source/workbook hashes are checked before reuse. Changed files are not overwritten. The workbook is a **raw dataset**, not a financial model: there are no formulas, formula caches or invented calculated results. All analytical outputs are recomputed in SQL.

In [2]:
generate_dossiers(BASE / 'dossiers')
paths = [export_excel(BASE / f'portfolios/claims_{size}', size) for size in sizes]
display(pd.DataFrame([{'claims': size, 'xlsx': str(path), 'bytes': path.stat().st_size}
                      for size, path in zip(sizes, paths, strict=True)]))

,claims,xlsx,bytes
0,10000,C:\Users\choun\Downloads\Prudential_Evidence_L...,545855
1,100000,C:\Users\choun\Downloads\Prudential_Evidence_L...,5334321


## 2. Inspect each worksheet
One claim = one row. Five claims share each policy. Dates are typed Excel dates, not strings displayed to resemble dates. Financial values use **integer EUR cents**. Claims!G2 = 15025000 means EUR 150,250.00, not EUR 15,025,000.

DossierLinks points to six local PDF dossiers. This is a locator table, not evidence that their conflicting declarations or invoices are correct. Expected failure labels remain outside the imported Claims and Policies tables.

In [3]:
from openpyxl import load_workbook
workbook = load_workbook(paths[0], read_only=True, data_only=False)
try:
    for sheet in workbook:
        rows = list(sheet.iter_rows(min_row=1, max_row=8, values_only=True))
        print('\nSHEET:', sheet.title)
        display(pd.DataFrame(rows[1:], columns=rows[0]))
finally:
    workbook.close()


SHEET: Readme


,Insurance Evidence Lab,Synthetic portfolio input
0,Purpose,Offline extraction and exhaustive SQL tests; n...
1,Declared claims,10000
2,Source,claims.jsonl; deterministic synthetic-claims-v...
3,Source SHA-256,76c99130d83b183ec9582fd87c9dda93859a2a34146563...
4,Money convention,"All amounts are integer EUR CENTS, not euros. ..."
5,Dates,"Actual Excel dates, displayed as yyyy-mm-dd. L..."
6,Claims,One row per claim. Five claims per policy. Exp...



SHEET: Claims


,claim_id,policy_id,tenant_id,loss_date,peril,status,claimed_cents,paid_cents,currency
0,SYN-C0000001,SYN-P0000001,SYNTHETIC_A,2025-01-01,water,open,15025000,0,EUR
1,SYN-C0000002,SYN-P0000001,SYNTHETIC_A,2025-02-02,theft,closed,32919,23043,EUR
2,SYN-C0000003,SYN-P0000001,SYNTHETIC_A,2025-03-03,fire,closed,40838,28586,EUR
3,SYN-C0000004,SYN-P0000001,SYNTHETIC_A,2025-04-04,storm,open,48757,0,EUR
4,SYN-C0000005,SYN-P0000001,SYNTHETIC_A,2025-05-05,water,closed,56676,39673,EUR
5,SYN-C0000006,SYN-P0000002,SYNTHETIC_B,2025-06-06,theft,closed,64595,45216,EUR
6,SYN-C0000007,SYN-P0000002,SYNTHETIC_B,2025-07-07,fire,open,72514,0,EUR



SHEET: Policies


,policy_id,tenant_id,effective_from,effective_to,terms_version
0,SYN-P0000001,SYNTHETIC_A,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
1,SYN-P0000002,SYNTHETIC_B,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
2,SYN-P0000003,SYNTHETIC_A,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
3,SYN-P0000004,SYNTHETIC_B,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
4,SYN-P0000005,SYNTHETIC_A,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
5,SYN-P0000006,SYNTHETIC_B,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1
6,SYN-P0000007,SYNTHETIC_A,2025-01-01,2025-12-31,SYNTHETIC-DEMO-1



SHEET: DossierLinks


,dossier,claim_id,policy_id,claims_excel_row,relative_folder
0,D001_consistent,SYN-C0000001,SYN-P0000001,2,../../dossiers/D001_consistent
1,D002_missing_invoice,SYN-C0000002,SYN-P0000001,3,../../dossiers/D002_missing_invoice
2,D003_wrong_policy,SYN-C0000003,SYN-P0000001,4,../../dossiers/D003_wrong_policy
3,D004_duplicate_invoice,SYN-C0000004,SYN-P0000001,5,../../dossiers/D004_duplicate_invoice
4,D005_amount_conflict,SYN-C0000005,SYN-P0000001,6,../../dossiers/D005_amount_conflict
5,D006_outside_period,SYN-C0000006,SYN-P0000002,7,../../dossiers/D006_outside_period



SHEET: DataDictionary


,field,type / unit,meaning
0,claim_id,unique text,Primary key of a synthetic claim; never a cust...
1,policy_id,text foreign key,Joins Claims to Policies; several claims share...
2,tenant_id,text scope,SYNTHETIC_A or SYNTHETIC_B. Not authenticated ...
3,loss_date,Excel date,Declared loss date; check against the matching...
4,peril,text enum,water / theft / fire / storm; artificial label...
5,status,text enum,open / closed; does not establish acceptance o...
6,claimed_cents,integer EUR cents,Synthetic claimed amount including deliberatel...


## 3. Verify every row, then compare exact scoped aggregates
The benchmark reads the actual XLSX. It rejects malformed inputs before returning a usable database, checks policy relationships, and switches SQLite to query-only mode. All row values must match the external JSONL source, and tenant totals must match the generator's expected totals.

Import timing includes XLSX reading, validation and SQL insertion; query timing covers the two scoped aggregate queries. Export time, row-by-row verification and workbook opening in Excel are not included. This is not a concurrent-load or production-memory benchmark.

In [4]:
results = [benchmark_excel(path.parent) for path in paths]
display(pd.DataFrame([{key: result[key] for key in [
    'rows','policy_rows','file_bytes','read_validate_import_seconds','query_seconds',
    'all_rows_equal','sql_matches_gold']} for result in results]))
assert all(r['all_rows_equal'] and r['sql_matches_gold'] for r in results)
for path, result in zip(paths, results, strict=True):
    (path.parent / 'xlsx_benchmark.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
display(pd.DataFrame([{'rows': r['rows'], 'tenant': tenant, **values}
                      for r in results for tenant, values in r['by_tenant'].items()]))

,rows,policy_rows,file_bytes,read_validate_import_seconds,query_seconds,all_rows_equal,sql_matches_gold
0,10000,2000,545855,4.920846,0.005929,True,True
1,100000,20000,5334321,49.483061,0.137777,True,True


,rows,tenant,count,claimed_cents,paid_cents
0,10000,SYNTHETIC_A,5000,5888215000,2724469159
1,10000,SYNTHETIC_B,5000,5862190000,2754796023
2,100000,SYNTHETIC_A,50000,58982150000,27504961159
3,100000,SYNTHETIC_B,50000,58966900000,27536279023


## 4. Link Excel rows to the actual dossier artifacts
Check all six links against claim ID, policy ID, tenant and physical source row. Also verify each dossier file hash. This proves the tested links and file identities, not authenticity or insurance coverage. Scenarios with a wrong policy in the PDF declaration remain intentional negative fixtures.

In [5]:
with closing(load_excel(paths[0])) as connection:
    links = verify_dossier_links(connection, paths[0], BASE / 'dossiers')
    display(pd.DataFrame(links))
    assert len(links) == 6 and all(row['link_verified'] for row in links)
    display(pd.DataFrame(connection.execute(
        'SELECT claim_id, source_name, source_row FROM claims ORDER BY source_row LIMIT 6'
    ).fetchall(), columns=['claim_id','source_workbook_and_sheet','physical_excel_row']))

,dossier,claim_id,excel_row,link_verified,coverage_decision
0,D001_consistent,SYN-C0000001,2,True,NOT_ASSESSED
1,D002_missing_invoice,SYN-C0000002,3,True,NOT_ASSESSED
2,D003_wrong_policy,SYN-C0000003,4,True,NOT_ASSESSED
3,D004_duplicate_invoice,SYN-C0000004,5,True,NOT_ASSESSED
4,D005_amount_conflict,SYN-C0000005,6,True,NOT_ASSESSED
5,D006_outside_period,SYN-C0000006,7,True,NOT_ASSESSED


,claim_id,source_workbook_and_sheet,physical_excel_row
0,SYN-C0000001,claims.xlsx:Claims,2
1,SYN-C0000002,claims.xlsx:Claims,3
2,SYN-C0000003,claims.xlsx:Claims,4
3,SYN-C0000004,claims.xlsx:Claims,5
4,SYN-C0000005,claims.xlsx:Claims,6
5,SYN-C0000006,claims.xlsx:Claims,7


## 5. Challenge the importer, not only the happy path
Each example creates a tiny independent workbook, corrupts one value, then attempts a real XLSX import. Formula cells are rejected even if a cached value could exist: openpyxl does not calculate Excel formulas, and stale caches are not authoritative source evidence. No paid LLM is used to repair bad data.

For this experiment, one bad row rejects the whole workbook. Production should add explicit quarantine and an operator review workflow; it should not silently drop rows and report an exhaustive total.

In [6]:
import sqlite3
cases = [
    ('formula', 'G3', '=SUM(1,2)'),
    ('missing amount', 'G3', None),
    ('mixed currency', 'I3', 'USD'),
    ('duplicate claim', 'A3', 'SYN-C0000001'),
    ('wrong tenant', 'C3', 'SYNTHETIC_B'),
    ('loss outside policy', 'D3', '2026-01-01'),
]
observations = []
with tempfile.TemporaryDirectory(prefix='insurance_xlsx_negative_') as temp:
    for index, (label, cell, value) in enumerate(cases):
        path = export_excel(Path(temp) / str(index), 10)
        book = load_workbook(path)
        try:
            book['Claims'][cell] = value
            book.save(path)
        finally:
            book.close()
        try:
            with closing(load_excel(path)):
                pass
        except (ValueError, sqlite3.IntegrityError) as error:
            observations.append({'case': label, 'rejected': True, 'reason': str(error)})
        else:
            observations.append({'case': label, 'rejected': False, 'reason': 'Unexpected acceptance'})
display(pd.DataFrame(observations))
assert all(item['rejected'] for item in observations)

,case,rejected,reason
0,formula,True,Claims!G3: formulas/errors not accepted
1,missing amount,True,Claims!G3: missing value
2,mixed currency,True,CHECK constraint failed: currency = 'EUR'
3,duplicate claim,True,UNIQUE constraint failed: claims.claim_id
4,wrong tenant,True,Policy relationship/scope/date mismatch for SY...
5,loss outside policy,True,Policy relationship/scope/date mismatch for SY...


## Interpretation and next boundary
Expected: both row counts round-trip exactly, scoped totals match, all six dossier links pass, and all six invalid examples are rejected. See the unit tests for more cases: missing sheets, headers, corrupt ZIPs, external links, invalid dates and changed-file protection.

This supports a concrete **Excel analytical ingestion** claim for the documented schema. It does not prove arbitrary Excel-layout understanding, formula evaluation, multi-currency accounting, real insurer applicability, authentication or million-row production readiness. Current bounds are 100,000 claims, 50 MiB compressed and 256 MiB expanded XLSX. Historical prudential files and notebooks are unchanged.

The importer is offline only. Do not upload client data to the public demo.